In [1]:
import pandas as pd

# Load the raw Ghana rainfall dataset (5-year-to-date CHIRPS aggregation)
df = pd.read_csv('../data/raw/gha-rainfall-subnat-5ytd.csv')

# Basic structural inspection
print("Shape:", df.shape)
print("\nColumn names:\n", df.columns.tolist())

# Check first few rows to catch any metadata/header artifacts common in HDX exports
print("\nFirst 3 rows:")
display(df.head(3))

# Inspect unique administrative regions (adjust column name if needed after seeing output)
if 'adm1_name' in df.columns:
    print("\nUnique adm1_name values:\n", df['adm1_name'].unique())
if 'adm2_name' in df.columns:
    print("\nUnique adm2_name values:\n", df['adm2_name'].unique())

Shape: (36675, 15)

Column names:
 ['date', 'adm_level', 'adm_id', 'PCODE', 'n_pixels', 'rfh', 'rfh_avg', 'r1h', 'r1h_avg', 'r3h', 'r3h_avg', 'rfq', 'r1q', 'r3q', 'version']

First 3 rows:


,date,adm_level,adm_id,PCODE,n_pixels,rfh,rfh_avg,r1h,r1h_avg,r3h,r3h_avg,rfq,r1q,r3q,version
0,2022-01-01,1,900854,GH02,798.0,1.631579,3.642774,7.929824,18.889140,244.740620,215.70810,76.72976,54.124283,113.15426,final
1,2022-01-11,1,900854,GH02,798.0,1.457394,2.760902,7.310777,12.792106,189.581470,156.99930,83.20416,69.192350,120.11254,final
2,2022-01-21,1,900854,GH02,798.0,1.486216,6.756349,4.575188,13.160025,118.897224,100.93584,55.17202,52.726734,116.95497,final


In [4]:
# Count how many adm2 (district-level) codes exist under each adm1 (region-level) prefix
# Each adm2 PCODE (e.g., GH0101) starts with its parent adm1 code (e.g., GH01)
district_counts = (
    df[df['adm_level'] == 2][['PCODE']]
    .drop_duplicates()
    .assign(adm1_prefix=lambda x: x['PCODE'].str[:4])  # e.g., 'GH01'
    .groupby('adm1_prefix')
    .size()
    .sort_values(ascending=False)
)

print("Number of districts per region code:")
print(district_counts)

# Greater Accra should show either 16 or 29 districts, depending on
# whether this dataset uses the older (16) or newer (29, post-2018 MMDA split) boundary set

Number of districts per region code:
adm1_prefix
GH02    30
GH06    26
GH05    20
GH14    17
GH08    15
GH07    15
GH12    13
GH15    13
GH03    11
GH13    11
GH04    10
GH16     9
GH10     8
GH01     6
GH11     6
GH09     5
dtype: int64


In [5]:
# Cross-check region identity using n_pixels, which scales with land area.
# Greater Accra is Ghana's smallest region (3,245 km²) by a wide margin,
# so it should have a distinctly small average pixel count vs all other regions.
region_area_proxy = (
    df[df['adm_level'] == 1]
    .groupby('PCODE')['n_pixels']
    .mean()
    .sort_values()
)

print("Average n_pixels per region (smallest = smallest land area):")
print(region_area_proxy)

Average n_pixels per region (smallest = smallest land area):
PCODE
GH07     115.0
GH12     286.0
GH05     322.0
GH06     545.0
GH14     592.0
GH13     622.0
GH15     792.0
GH02     798.0
GH04    1249.0
GH11    2257.0
Name: n_pixels, dtype: float64


In [6]:
# Filter to Greater Accra Region only, using the verified PCODE 'GH07'
# We take district-level (adm_level 2) rows for granularity across Accra's sub-units,
# giving us more data points (multiple locations x time) than the single regional aggregate.
accra_df = df[(df['adm_level'] == 2) & (df['PCODE'].str.startswith('GH07'))].copy()

print("Filtered shape:", accra_df.shape)
print("Unique sub-region PCODEs in Accra subset:", accra_df['PCODE'].nunique())
print("Date range:", accra_df['date'].min(), "to", accra_df['date'].max())

# Engineer a binary proxy flood risk label:
# Flag extreme positive rainfall anomalies as high flood risk.
# r1q/r3q are % anomalies vs long-term average; >150% sustained anomaly signals
# unusually saturated ground conditions consistent with flood risk.
accra_df['Flood_Risk_Label'] = (
    (accra_df['r1q'] > 150) & (accra_df['r3q'] > 130)
).astype(int)

print("\nFlood_Risk_Label distribution:")
print(accra_df['Flood_Risk_Label'].value_counts(normalize=True))

Filtered shape: (2445, 15)
Unique sub-region PCODEs in Accra subset: 15
Date range: 2022-01-01 to 2026-07-01

Flood_Risk_Label distribution:
Flood_Risk_Label
0    0.837219
1    0.162781
Name: proportion, dtype: float64


In [7]:
from sklearn.model_selection import train_test_split

# Define predictive numerical features and isolate the target
feature_cols = ['rfh', 'r1h', 'r3h', 'rfq', 'r1q', 'r3q']
X = accra_df[feature_cols].copy()
y = accra_df['Flood_Risk_Label'].copy()

# 80/20 train/test split, stratified to preserve the ~84/16 class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("\nTrain label distribution:")
print(y_train.value_counts(normalize=True))
print("\nTest label distribution:")
print(y_test.value_counts(normalize=True))

X_train shape: (1956, 6)
X_test shape: (489, 6)

Train label distribution:
Flood_Risk_Label
0    0.837423
1    0.162577
Name: proportion, dtype: float64

Test label distribution:
Flood_Risk_Label
0    0.836401
1    0.163599
Name: proportion, dtype: float64


In [8]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Initialize XGBoost classifier
# scale_pos_weight helps compensate for class imbalance (~5.1:1 ratio of 0s to 1s)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_clf = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42
)

# Train the model
xgb_clf.fit(X_train, y_train)

# Generate predictions on the held-out test set
y_pred = xgb_clf.predict(X_test)

# Evaluate performance
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       409
           1       1.00      1.00      1.00        80

    accuracy                           1.00       489
   macro avg       1.00      1.00      1.00       489
weighted avg       1.00      1.00      1.00       489

Confusion Matrix:
[[409   0]
 [  0  80]]


In [9]:
# --- Feature selection (updated: exclude r1q, r3q to avoid label leakage) ---
feature_cols = ['rfh', 'r1h', 'r3h', 'rfq']
X = accra_df[feature_cols].copy()
y = accra_df['Flood_Risk_Label'].copy()

print("Feature columns:", feature_cols)
print("X shape:", X.shape)

Feature columns: ['rfh', 'r1h', 'r3h', 'rfq']
X shape: (2445, 4)


In [10]:
from sklearn.model_selection import train_test_split

# 80/20 train/test split, stratified to preserve class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1956, 4)
X_test shape: (489, 4)


In [11]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Account for class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_clf = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42
)

xgb_clf.fit(X_train, y_train)

y_pred = xgb_clf.predict(X_test)

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.90      0.92       409
           1       0.59      0.72      0.65        80

    accuracy                           0.87       489
   macro avg       0.77      0.81      0.79       489
weighted avg       0.89      0.87      0.88       489

Confusion Matrix:
[[369  40]
 [ 22  58]]


In [12]:
import os
import joblib

# Export the trained model to disk
output_dir = 'ml_models/artifacts'
os.makedirs(output_dir, exist_ok=True)

model_path = os.path.join(output_dir, 'accra_flood_risk_xgb.joblib')
joblib.dump(xgb_clf, model_path)

print(f"Model saved to: {model_path}")

Model saved to: ml_models/artifacts/accra_flood_risk_xgb.joblib
